# CeNNMixer-v2 — Progressive Qwen3.5 Mixer Takeover

This version fixes the main CeNNMixer-v1 training problem.

Instead of deleting the Qwen mixer immediately, v2 trains CeNN **beside** the frozen Qwen mixer:

[
y = (1-alpha)y_{Qwen} + alpha y_{CeNN}
]

Schedule:

    α = 0.00  -> externally exact Qwen, CeNN learns local mixer output
    α = 0.05
    α = 0.10
    α = 0.25
    α = 0.50
    α = 0.75
    α = 1.00  -> CeNN has full control

At every stage:
- CeNN is directly trained against the frozen Qwen mixer output
- validation runs repeatedly
- the best checkpoint is saved
- the best checkpoint is restored before the next α
- training extends automatically if the stage has not converged
- LR is reduced on plateaus

After α=1, the original Qwen mixer is physically removed and the CeNN-only result is verified again.

Qwen embedding, RMSNorm, FFN and LM head stay unchanged.


In [ ]:
#@title 1. Setup
import pathlib, subprocess, sys, importlib, json, torch, shutil, os

REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers','accelerate','datasets','pandas','matplotlib',
    'huggingface_hub','safetensors'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC=REPO_DIR/'src'
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
for n in list(sys.modules):
    if n=='tinycenn_lm' or n.startswith('tinycenn_lm.'):
        del sys.modules[n]
importlib.invalidate_caches()

for p in [
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_cennmixer_v2.py',
    REPO_DIR/'scripts'/'run_qwen35_cennmixer_v2.py',
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

print('✓ CeNNMixer-v2 preflight OK')
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
else:
    print('⚠️ Select a GPU runtime.')


In [ ]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
LAYERS='0' #@param {type:'string'}

ALPHAS='0,0.05,0.10,0.25,0.50,0.75,1.0' #@param {type:'string'}

SEQ_LEN=128 #@param {type:'integer'}
TRAIN_BLOCKS=768 #@param {type:'integer'}
VAL_BLOCKS=32 #@param {type:'integer'}

# v2 starts larger than v1 intentionally: first prove replacement quality,
# then reduce CeNN capacity in a later experiment.
GROUPS=32 #@param {type:'integer'}
CELL_DIM=48 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}

LR=0.0002 #@param {type:'number'}
STAGE_UPDATES=300 #@param {type:'integer'}
EXTEND_UPDATES=200 #@param {type:'integer'}
MAX_STAGE_UPDATES=3000 #@param {type:'integer'}
PROBE_EVERY=50 #@param {type:'integer'}
PATIENCE_PROBES=8 #@param {type:'integer'}
MIN_LR=0.0000125 #@param {type:'number'}

TOPK=64 #@param {type:'integer'}
MIN_TOP1=0.97 #@param {type:'number'}
MAX_KL=0.03 #@param {type:'number'}
MAX_HIDDEN_MSE=0.05 #@param {type:'number'}
MAX_MIXER_MSE=0.12 #@param {type:'number'}

OUTPUT_DIR=REPO_DIR/'results'/'cennmixer_v2_qwen35_08b'

print('Layers:',LAYERS)
print('Alpha schedule:',ALPHAS)
print('CeNN groups × cell_dim:',GROUPS,'×',CELL_DIM)
print('State per timescale:',GROUPS*CELL_DIM)
print('LR:',LR)
print('Max updates per alpha:',MAX_STAGE_UPDATES)


In [ ]:
#@title 3. Train CeNNMixer-v2
cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_cennmixer_v2.py'),
    '--base-model',BASE_MODEL,
    '--layers',LAYERS,
    '--alphas',ALPHAS,
    '--seq-len',str(SEQ_LEN),
    '--train-blocks',str(TRAIN_BLOCKS),
    '--val-blocks',str(VAL_BLOCKS),
    '--groups',str(GROUPS),
    '--cell-dim',str(CELL_DIM),
    '--graph-steps',str(GRAPH_STEPS),
    '--lr',str(LR),
    '--stage-updates',str(STAGE_UPDATES),
    '--extend-updates',str(EXTEND_UPDATES),
    '--max-stage-updates',str(MAX_STAGE_UPDATES),
    '--probe-every',str(PROBE_EVERY),
    '--patience-probes',str(PATIENCE_PROBES),
    '--min-lr',str(MIN_LR),
    '--topk',str(TOPK),
    '--min-top1',str(MIN_TOP1),
    '--max-kl',str(MAX_KL),
    '--max-hidden-mse',str(MAX_HIDDEN_MSE),
    '--max-mixer-mse',str(MAX_MIXER_MSE),
    '--output-dir',str(OUTPUT_DIR),
]

print('='*120)
print(' '.join(cmd))
print('='*120)

p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc:
    raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Stage summary
import pandas as pd
from IPython.display import display

report=json.loads((OUTPUT_DIR/'report.json').read_text())
stage=pd.read_csv(OUTPUT_DIR/'stage_summary.csv')
hist=pd.read_csv(OUTPUT_DIR/'training_history.csv')

print('Architecture:',report['architecture'])
print('Layers:',report['layers'])
print('Original types:',report['layer_kinds'])
print('CeNN params:',f"{report['cenn_params']:,}")
print('Qwen mixer params replaced:',f"{report['replaced_qwen_mixer_params']:,}")
print('Mixer parameter reduction:',f"{report['mixer_param_reduction_pct']:.2f}%")
print('STRICT FINAL QUALITY GATE:',report['strict_quality_gate'])

display(stage[[
    'alpha','best_step','trained_steps','pass','violation',
    'student_ce','ce_gap','kl','hidden_mse','delta_mse','mixer_mse','top1',
    'generation_exact_rate','generation_mean_jaccard'
]])


In [ ]:
#@title 5. Progressive takeover plots
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['top1'],marker='o')
plt.axhline(MIN_TOP1,linestyle='--')
plt.xlabel('alpha: CeNN control')
plt.ylabel('top-1 agreement')
plt.title('CeNN takeover — top-1 agreement with Qwen')
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['kl'],marker='o')
plt.axhline(MAX_KL,linestyle='--')
plt.xlabel('alpha')
plt.ylabel('KL')
plt.title('CeNN takeover — output distribution drift')
plt.show()

plt.figure(figsize=(9,5))
plt.plot(stage['alpha'],stage['hidden_mse'],marker='o',label='hidden MSE')
plt.plot(stage['alpha'],stage['mixer_mse'],marker='o',label='direct mixer MSE')
plt.xlabel('alpha')
plt.ylabel('relative error')
plt.legend()
plt.title('CeNN takeover — representation matching')
plt.show()


In [ ]:
#@title 6. Training history
display(hist.tail(30))

plt.figure(figsize=(10,5))
for a in sorted(hist['alpha'].unique()):
    d=hist[hist['alpha']==a]
    plt.plot(d['stage_step'],d['mixer_mse'],marker='o',label=f'α={a:g}')
plt.xlabel('stage step')
plt.ylabel('direct mixer MSE')
plt.title('Direct Qwen-mixer → CeNN distillation')
plt.legend()
plt.show()


In [ ]:
#@title 7. Final α=1 and CeNN-only verification
print('α=1 before physically removing Qwen mixer:')
print(json.dumps(report['alpha1_before_removing_qwen_mixer'],indent=2))

print('\nAfter physically removing the Qwen mixer:')
print(json.dumps(report['final_cenn_only_probe'],indent=2))

print('\nGeneration after Qwen mixer removal:')
for i,x in enumerate(report['final_cenn_only_generation'],1):
    print('\n'+'='*100)
    print(i,'USER:',x['prompt'])
    print('QWEN:',x['qwen'])
    print('CeNN-v2:',x['cenn'])
    print('exact=',x['exact'],'jaccard=',round(x['jaccard'],3))


## What success looks like

The important difference from v1 is **not CE alone**.

A good α=1 result should have all of these moving together:

    top1          -> 0.97 or higher
    KL            -> 0.03 or lower
    hidden MSE    -> 0.05 or lower
    direct mixer MSE low
    generation coherent

If α=0 learns the mixer well but α=0.50 or α=0.75 stops converging, that tells us the CeNN architecture still lacks part of DeltaNet's sequence function.

If α=1 works, then the original Qwen mixer can be removed and we can next reduce GROUPS/CELL_DIM to find the smallest CeNN that preserves quality.


In [ ]:
#@title 8. Upload final CeNNMixer-v2 adapter to Hugging Face
from huggingface_hub import HfApi, login
import shutil, json, os

HF_REPO_ID='vtava/Qwen35-0.8B-CeNNMixer-v2' #@param {type:'string'}
HF_PRIVATE=False #@param {type:'boolean'}

HF_EXPORT=REPO_DIR/'results'/'Qwen35-0.8B-CeNNMixer-v2-HF'
if HF_EXPORT.exists():
    shutil.rmtree(HF_EXPORT)
HF_EXPORT.mkdir(parents=True,exist_ok=True)

for name in [
    'cennmixer_v2_final_cenn_only.pt',
    'report.json',
    'stage_summary.csv',
    'training_history.csv',
]:
    src=OUTPUT_DIR/name
    if src.exists():
        shutil.copy2(src,HF_EXPORT/name)

cfg={
    'base_model':report['base_model'],
    'architecture':report['architecture'],
    'layers':report['layers'],
    'layer_kinds':report['layer_kinds'],
    'alpha_schedule':report['alpha_schedule'],
    'cenn_config':report['config'],
    'strict_quality_gate':report['strict_quality_gate'],
    'final_metrics':report['final_cenn_only_probe'],
}
(HF_EXPORT/'cennmixer_v2_config.json').write_text(json.dumps(cfg,indent=2),encoding='utf-8')

m=report['final_cenn_only_probe']
readme=f"""---
base_model: {report['base_model']}
library_name: transformers
pipeline_tag: text-generation
tags:
- qwen
- cenn
- recurrent
- sequence-mixer
- tinycenn
---

# Qwen3.5-0.8B CeNNMixer-v2

Progressive CeNN replacement experiment for Qwen3.5 sequence mixers.

## Replaced layers
{report['layers']}

Original mixer types:
{json.dumps(report['layer_kinds'],indent=2)}

## Progressive takeover
Alpha schedule: {report['alpha_schedule']}

The frozen Qwen mixer remains active during distillation while CeNN progressively
takes control. At alpha=1 the original mixer is removed.

## Final CeNN-only metrics
- Top-1 agreement: {m['top1']:.6f}
- KL vs Qwen: {m['kl']:.6f}
- Hidden MSE: {m['hidden_mse']:.6f}
- Student CE: {m['student_ce']:.6f}
- Teacher CE: {m['teacher_ce']:.6f}
- Strict quality gate: {report['strict_quality_gate']}

## Parameters
- CeNN: {report['cenn_params']:,}
- Qwen mixer replaced: {report['replaced_qwen_mixer_params']:,}
- Mixer parameter reduction: {report['mixer_param_reduction_pct']:.2f}%

Qwen embedding, RMSNorm, FFN and LM head remain unchanged.

Project: https://github.com/vtavakkoli/TinyCeNN-LM
"""
(HF_EXPORT/'README.md').write_text(readme,encoding='utf-8')

token=None
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
except Exception:
    pass

if token:
    login(token=token,add_to_git_credential=False)
else:
    print('No HF_TOKEN secret found. Enter a Hugging Face WRITE token.')
    login()

api=HfApi()
api.create_repo(HF_REPO_ID,repo_type='model',private=HF_PRIVATE,exist_ok=True)
api.upload_folder(
    folder_path=str(HF_EXPORT),
    repo_id=HF_REPO_ID,
    repo_type='model',
    commit_message='Upload CeNNMixer-v2 final CeNN-only adapter and results',
)
print('✓ Uploaded: https://huggingface.co/'+HF_REPO_ID)
